In [ ]:
!pip install -q transformers accelerate

In [ ]:
import os
import torch

files_to_check = [
    "/content/coco5000_metadata.json",
    "/content/coco1000_image_ids.json",
    "/content/coco5000_image_features.pt",
    "/content/coco5000_text_features.pt",
    "/content/coco5000_caption_image_indices.json",
]

for path in files_to_check:
    print(path, os.path.exists(path))

print("CUDA:", torch.cuda.is_available())

In [ ]:
%cd /content

!wget -O val2017.zip -q --show-progress http://images.cocodataset.org/zips/val2017.zip
!unzip -o -q val2017.zip

print("图片数量:")
!find /content/val2017 -maxdepth 1 -name "*.jpg" | wc -l

In [ ]:
import json
import random

random.seed(123)

# 读取完整 metadata
with open(
    "/content/coco5000_metadata.json",
    "r",
    encoding="utf-8"
) as f:
    metadata_5000 = json.load(f)

# 读取固定测试集编号
with open(
    "/content/coco1000_image_ids.json",
    "r",
    encoding="utf-8"
) as f:
    test_image_ids = json.load(f)

test_id_set = set(test_image_ids)

test_metadata = []
train_pool = []

for item in metadata_5000:
    if item["image_id"] in test_id_set:
        test_metadata.append(item)
    else:
        train_pool.append(item)

train_pool_ids = [
    item["image_id"]
    for item in train_pool
]

val_id_set = set(
    random.sample(train_pool_ids, 500)
)

val_metadata = [
    item
    for item in train_pool
    if item["image_id"] in val_id_set
]

print("验证图片:", len(val_metadata))
print("测试图片:", len(test_metadata))

val_caption_count = sum(
    len(item["captions"])
    for item in val_metadata
)

test_caption_count = sum(
    len(item["captions"])
    for item in test_metadata
)

print("验证描述:", val_caption_count)
print("测试描述:", test_caption_count)

with open(
    "/content/coco500_val_metadata.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(val_metadata, f, ensure_ascii=False, indent=2)

with open(
    "/content/coco1000_test_metadata.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(test_metadata, f, ensure_ascii=False, indent=2)

print("已保存验证集和测试集 metadata")

In [ ]:
import json
import random

random.seed(123)

with open(
    "/content/coco5000_metadata.json",
    "r",
    encoding="utf-8"
) as f:
    metadata_5000 = json.load(f)

with open(
    "/content/coco1000_image_ids.json",
    "r",
    encoding="utf-8"
) as f:
    test_ids = set(json.load(f))

test_metadata = []
train_pool = []

for item in metadata_5000:
    if item["image_id"] in test_ids:
        test_metadata.append(item)
    else:
        train_pool.append(item)

# 从剩余 4000 张中固定抽 500 张验证集
train_pool_ids = [
    item["image_id"]
    for item in train_pool
]

val_ids = set(
    random.sample(train_pool_ids, 500)
)

val_metadata = []
train_metadata = []

for item in train_pool:
    if item["image_id"] in val_ids:
        val_metadata.append(item)
    else:
        train_metadata.append(item)

with open(
    "/content/coco500_val_metadata.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(val_metadata, f, ensure_ascii=False, indent=2)

with open(
    "/content/coco1000_test_metadata.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(test_metadata, f, ensure_ascii=False, indent=2)

with open(
    "/content/coco3500_train_metadata.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(train_metadata, f, ensure_ascii=False, indent=2)

print("训练集:", len(train_metadata))
print("验证集:", len(val_metadata))
print("测试集:", len(test_metadata))

In [ ]:
import json
import csv
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
top_k = 20

# 读取完整 metadata
with open(
    "/content/coco5000_metadata.json",
    "r",
    encoding="utf-8"
) as f:
    metadata_5000 = json.load(f)

# 读取验证集和测试集
with open(
    "/content/coco500_val_metadata.json",
    "r",
    encoding="utf-8"
) as f:
    val_metadata = json.load(f)

with open(
    "/content/coco1000_test_metadata.json",
    "r",
    encoding="utf-8"
) as f:
    test_metadata = json.load(f)

# 读取描述到图片的映射
with open(
    "/content/coco5000_caption_image_indices.json",
    "r",
    encoding="utf-8"
) as f:
    caption_image_indices = json.load(f)

# 加载 CLIP 向量
image_features = torch.load(
    "/content/coco5000_image_features.pt",
    map_location="cpu"
).to(device)

text_features = torch.load(
    "/content/coco5000_text_features.pt",
    map_location="cpu"
).to(device)

# 图片编号 -> 完整 5000 张中的索引
image_id_to_full_index = {
    item["image_id"]: index
    for index, item in enumerate(metadata_5000)
}


def compute_clip_candidates(split_metadata, split_name):
    """计算一个 split 的 CLIP Top-K 候选和 baseline。"""
    image_id_to_local_index = {
        item["image_id"]: index
        for index, item in enumerate(split_metadata)
    }

    image_id_set = set(image_id_to_local_index.keys())

    image_indices = [
        image_id_to_full_index[item["image_id"]]
        for item in split_metadata
    ]

    caption_positions = []
    caption_local_image_indices = []

    for full_position, full_image_index in enumerate(caption_image_indices):
        image_id = metadata_5000[full_image_index]["image_id"]

        if image_id not in image_id_set:
            continue

        caption_positions.append(full_position)
        caption_local_image_indices.append(
            image_id_to_local_index[image_id]
        )

    split_image_features = image_features[image_indices]
    split_text_features = text_features[caption_positions]

    with torch.no_grad():
        similarity = split_image_features @ split_text_features.T

        topk_positions = similarity.topk(
            top_k,
            dim=1
        ).indices.cpu().tolist()

    mapping = torch.tensor(
        caption_local_image_indices,
        device=device
    )

    num_images = len(split_metadata)
    correct_indices = torch.arange(
        num_images,
        device=device
    ).unsqueeze(1)

    metrics = {}

    for k in [1, 5, 10]:
        topk_caption_indices = torch.tensor(
            topk_positions,
            device=device
        )[:, :k]

        topk_image_indices = mapping[topk_caption_indices]

        correct_any = (
            topk_image_indices == correct_indices
        ).any(dim=1)

        metrics[k] = correct_any.float().mean().item()

    # 保存候选：记录每条候选在完整描述列表中的位置
    candidates = []

    for item, positions in zip(split_metadata, topk_positions):
        global_positions = [
            caption_positions[position]
            for position in positions
        ]

        candidates.append({
            "image_id": item["image_id"],
            "file_name": item["file_name"],
            "candidate_positions": global_positions,
        })

    with open(
        f"/content/{split_name}_candidates_top20.json",
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(candidates, f, ensure_ascii=False, indent=2)

    with open(
        f"/content/{split_name}_baseline_metrics.csv",
        "w",
        newline="",
        encoding="utf-8"
    ) as f:
        writer = csv.writer(f)
        writer.writerow(["metric", "value"])
        for k in [1, 5, 10]:
            writer.writerow([f"Recall@{k}", metrics[k]])

    print(f"\n{split_name}")
    print("图片数量:", num_images)
    print("候选描述数量:", len(caption_positions))
    print("Recall@1:", f"{metrics[1]:.4f}")
    print("Recall@5:", f"{metrics[5]:.4f}")
    print("Recall@10:", f"{metrics[10]:.4f}")
    print("候选文件已保存")

    return metrics


val_metrics = compute_clip_candidates(
    val_metadata,
    "coco500_val"
)

test_metrics = compute_clip_candidates(
    test_metadata,
    "coco1000_test"
)

In [ ]:
import json
import torch
from PIL import Image
from transformers import BlipProcessor, BlipForImageTextRetrieval

device = "cuda" if torch.cuda.is_available() else "cpu"

# 加载 BLIP 图文匹配模型
processor = BlipProcessor.from_pretrained(
    "Salesforce/blip-itm-base-coco"
)

model = BlipForImageTextRetrieval.from_pretrained(
    "Salesforce/blip-itm-base-coco"
).to(device)

model.eval()

print("BLIP 模型加载完成")

In [ ]:
import json
from PIL import Image
import torch

# 读取验证集候选
with open(
    "/content/coco500_val_candidates_top20.json",
    "r",
    encoding="utf-8"
) as f:
    val_candidates = json.load(f)

# 读取完整 metadata
with open(
    "/content/coco5000_metadata.json",
    "r",
    encoding="utf-8"
) as f:
    metadata_5000 = json.load(f)

with open(
    "/content/coco5000_caption_image_indices.json",
    "r",
    encoding="utf-8"
) as f:
    caption_image_indices = json.load(f)

# 先取第一张验证图片
item = val_candidates[0]

image_id = item["image_id"]
file_name = item["file_name"]
candidate_positions = item["candidate_positions"]

# 找到这张图片对应的原始图片路径
image_path = f"/content/val2017/{file_name}"

# 找到这张图片的正确描述
correct_captions = []
for position, full_image_index in enumerate(caption_image_indices):
    if metadata_5000[full_image_index]["image_id"] == image_id:
        correct_captions.append(
            metadata_5000[full_image_index]["captions"]
        )
        break

# 用全局描述位置取出 20 条候选文字
all_captions = []
for metadata_item in metadata_5000:
    all_captions.extend(metadata_item["captions"])

candidate_captions = [
    all_captions[position]
    for position in candidate_positions
]

print("图片:", file_name)
print("候选描述数量:", len(candidate_captions))
print("第一条候选:", candidate_captions[0])

# BLIP 要求图片和文字 batch 对齐
image = Image.open(image_path).convert("RGB")
images = [image] * len(candidate_captions)

inputs = processor(
    images=images,
    text=candidate_captions,
    return_tensors="pt",
    padding=True,
).to(device)

with torch.no_grad():
    outputs = model(**inputs)

print("输出类型:", type(outputs))
print("输出字段:", outputs.keys())

# 不同 transformers 版本可能返回 itm_score 或第 0 项
if hasattr(outputs, "itm_score"):
    itm_scores = outputs.itm_score
else:
    itm_scores = outputs[0]

print("itm_scores shape:", itm_scores.shape)

# 第二列通常表示匹配分数
scores = itm_scores[:, 1]

# 按 BLIP 分数排序
ranked_indices = torch.argsort(
    scores,
    descending=True
).cpu().tolist()

print("\nBLIP 排序前 5:")
for rank, index in enumerate(ranked_indices[:5], start=1):
    print(
        f"{rank}. score={scores[index].item():.4f} "
        f"caption={candidate_captions[index]}"
    )

In [ ]:
import json
import csv
import torch
from PIL import Image

# 读取验证集题目
with open(
    "/content/coco500_val_metadata.json",
    "r",
    encoding="utf-8"
) as f:
    val_metadata = json.load(f)

# 读取 CLIP Top-20 候选
with open(
    "/content/coco500_val_candidates_top20.json",
    "r",
    encoding="utf-8"
) as f:
    val_candidates = json.load(f)

# 读取完整 metadata 和描述映射
with open(
    "/content/coco5000_metadata.json",
    "r",
    encoding="utf-8"
) as f:
    metadata_5000 = json.load(f)

with open(
    "/content/coco5000_caption_image_indices.json",
    "r",
    encoding="utf-8"
) as f:
    caption_image_indices = json.load(f)

# 把所有描述展开，方便通过全局位置取文字
all_captions = []
for item in metadata_5000:
    all_captions.extend(item["captions"])

# 验证图片编号集合
val_image_ids = {
    item["image_id"]
    for item in val_metadata
}

# 只保留验证集的候选
val_candidates = [
    item
    for item in val_candidates
    if item["image_id"] in val_image_ids
]

print("验证图片数量:", len(val_candidates))

reranked_correct = {
    1: 0,
    5: 0,
    10: 0,
}

reranked_results = []

model.eval()

for index, item in enumerate(val_candidates):
    image_id = item["image_id"]
    file_name = item["file_name"]
    candidate_positions = item["candidate_positions"]

    image_path = f"/content/val2017/{file_name}"
    image = Image.open(image_path).convert("RGB")

    candidate_captions = [
        all_captions[position]
        for position in candidate_positions
    ]

    # BLIP 需要图片和文字数量一致
    images = [image] * len(candidate_captions)

    inputs = processor(
        images=images,
        text=candidate_captions,
        return_tensors="pt",
        padding=True,
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        scores = outputs.itm_score[:, 1]

    # 按 BLIP 分数从高到低排序
    ranked_indices = torch.argsort(
        scores,
        descending=True
    ).cpu().tolist()

    # 每条候选描述是否属于当前图片
    correct_flags = [
        metadata_5000[
            caption_image_indices[position]
        ]["image_id"] == image_id
        for position in candidate_positions
    ]

    # 计算重排序后的 Recall@1/5/10
    for k in [1, 5, 10]:
        topk_ranked_indices = ranked_indices[:k]

        if any(correct_flags[i] for i in topk_ranked_indices):
            reranked_correct[k] += 1

    reranked_results.append({
        "image_id": image_id,
        "file_name": file_name,
        "candidate_positions": candidate_positions,
        "ranked_indices": ranked_indices,
        "blip_scores": scores.cpu().tolist(),
        "correct_flags": correct_flags,
    })

    if (index + 1) % 25 == 0 or index + 1 == len(val_candidates):
        print(f"重排序完成: {index + 1}/{len(val_candidates)}")

num_images = len(val_candidates)

metrics = {
    k: reranked_correct[k] / num_images
    for k in [1, 5, 10]
}

print("\n========== BLIP 重排序验证集结果 ==========")
for k in [1, 5, 10]:
    print(f"Recall@{k}: {metrics[k]:.4f}")

baseline = {
    1: 0.7840,
    5: 0.9580,
    10: 0.9860,
}

print("\n========== 与 CLIP baseline 对比 ==========")
for k in [1, 5, 10]:
    print(
        f"Recall@{k}: "
        f"baseline={baseline[k]:.4f}, "
        f"rerank={metrics[k]:.4f}, "
        f"delta={metrics[k] - baseline[k]:+.4f}"
    )

# 保存指标
with open(
    "/content/coco500_val_rerank_metrics.csv",
    "w",
    newline="",
    encoding="utf-8"
) as f:
    writer = csv.writer(f)
    writer.writerow(["metric", "baseline", "rerank", "delta"])

    for k in [1, 5, 10]:
        writer.writerow([
            f"Recall@{k}",
            baseline[k],
            metrics[k],
            metrics[k] - baseline[k],
        ])

# 保存重排序明细
with open(
    "/content/coco500_val_rerank_results.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(reranked_results, f, ensure_ascii=False)

print("\n已保存:")
print("/content/coco500_val_rerank_metrics.csv")
print("/content/coco500_val_rerank_results.json")

In [ ]:
import json
import csv
import torch
from PIL import Image

# 读取测试集
with open(
    "/content/coco1000_test_metadata.json",
    "r",
    encoding="utf-8"
) as f:
    test_metadata = json.load(f)

# 读取测试集 CLIP Top-20 候选
with open(
    "/content/coco1000_test_candidates_top20.json",
    "r",
    encoding="utf-8"
) as f:
    test_candidates = json.load(f)

# 读取完整 metadata 和描述映射
with open(
    "/content/coco5000_metadata.json",
    "r",
    encoding="utf-8"
) as f:
    metadata_5000 = json.load(f)

with open(
    "/content/coco5000_caption_image_indices.json",
    "r",
    encoding="utf-8"
) as f:
    caption_image_indices = json.load(f)

# 展开全部描述
all_captions = []
for item in metadata_5000:
    all_captions.extend(item["captions"])

test_image_ids = {
    item["image_id"]
    for item in test_metadata
}

test_candidates = [
    item
    for item in test_candidates
    if item["image_id"] in test_image_ids
]

print("测试图片数量:", len(test_candidates))

reranked_correct = {
    1: 0,
    5: 0,
    10: 0,
}

reranked_results = []

model.eval()

for index, item in enumerate(test_candidates):
    image_id = item["image_id"]
    file_name = item["file_name"]
    candidate_positions = item["candidate_positions"]

    image_path = f"/content/val2017/{file_name}"
    image = Image.open(image_path).convert("RGB")

    candidate_captions = [
        all_captions[position]
        for position in candidate_positions
    ]

    images = [image] * len(candidate_captions)

    inputs = processor(
        images=images,
        text=candidate_captions,
        return_tensors="pt",
        padding=True,
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        scores = outputs.itm_score[:, 1]

    ranked_indices = torch.argsort(
        scores,
        descending=True
    ).cpu().tolist()

    correct_flags = [
        metadata_5000[
            caption_image_indices[position]
        ]["image_id"] == image_id
        for position in candidate_positions
    ]

    for k in [1, 5, 10]:
        topk_ranked_indices = ranked_indices[:k]

        if any(correct_flags[i] for i in topk_ranked_indices):
            reranked_correct[k] += 1

    reranked_results.append({
        "image_id": image_id,
        "file_name": file_name,
        "candidate_positions": candidate_positions,
        "ranked_indices": ranked_indices,
        "blip_scores": scores.cpu().tolist(),
        "correct_flags": correct_flags,
    })

    if (index + 1) % 50 == 0 or index + 1 == len(test_candidates):
        print(f"测试重排序完成: {index + 1}/{len(test_candidates)}")

num_images = len(test_candidates)

metrics = {
    k: reranked_correct[k] / num_images
    for k in [1, 5, 10]
}

baseline = {
    1: 0.7020,
    5: 0.8980,
    10: 0.9520,
}

print("\n========== 1000 张测试集最终结果 ==========")

for k in [1, 5, 10]:
    print(f"Recall@{k}: {metrics[k]:.4f}")

print("\n========== 与 CLIP baseline 对比 ==========")

for k in [1, 5, 10]:
    print(
        f"Recall@{k}: "
        f"baseline={baseline[k]:.4f}, "
        f"rerank={metrics[k]:.4f}, "
        f"delta={metrics[k] - baseline[k]:+.4f}"
    )

with open(
    "/content/coco1000_test_rerank_metrics.csv",
    "w",
    newline="",
    encoding="utf-8"
) as f:
    writer = csv.writer(f)
    writer.writerow(["metric", "baseline", "rerank", "delta"])

    for k in [1, 5, 10]:
        writer.writerow([
            f"Recall@{k}",
            baseline[k],
            metrics[k],
            metrics[k] - baseline[k],
        ])

with open(
    "/content/coco1000_test_rerank_results.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(reranked_results, f, ensure_ascii=False)

print("\n已保存:")
print("/content/coco1000_test_rerank_metrics.csv")
print("/content/coco1000_test_rerank_results.json")

In [ ]:
from google.colab import files

files.download("/content/coco1000_test_rerank_metrics.csv")
files.download("/content/coco1000_test_rerank_results.json")
files.download("/content/coco500_val_rerank_metrics.csv")
files.download("/content/coco500_val_rerank_results.json")
